# 08 — Massimizzare DHSLP (pruned + regolarizzazione)
DHSLP oggi è debole (dep 0.24, mixed 0.21) e **overfitta** (train→1.0, test chance). Qui proviamo a
spingerlo verso il tetto delle ConvNet (~0.5) con tre leve:
1. **ipergrafo pruned** da connettività PCC/PLV (struttura fissa data-driven, meno parametri da overfittare),
2. **hybrid** (appreso + pruned),
3. **regolarizzazione** (dropout, weight decay, label smoothing, modello più piccolo).

> ⚠️ **Aspettativa onesta**: la connettività è cieca alla parola (word NMI=0.001) → l'ipergrafo pruned
> difficilmente sblocca la discriminazione. Obiettivo realistico: ridurre l'overfitting e avvicinarsi
> alle ConvNet, con un risultato 'DHSLP massimizzato' pulito per la tesi (anche se resta sotto Shallow).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import track3_config as C, track3_train as T
print(C.summary()); assert C.DATA_ROOT is not None, C._no_data_msg()

## §1 — Sweep rapido su 5 soggetti (trova la config migliore)
Ogni config su S01–S05 (subject-dependent, 100 epoche). Guarda **test** e **train** (gap = overfitting).

In [ ]:
BASE_TK = dict(epochs=100, patience=20, lr=1e-3, batch_size=32)
CONFIGS = [
    ('learned (baseline)', dict(hyperedge_mode='learned'), {}),
    ('learned+reg',        dict(hyperedge_mode='learned', dropout=0.7), dict(weight_decay=1e-2, label_smoothing=0.1)),
    ('pruned_pcc_k8',      dict(hyperedge_mode='pruned', metric='pcc', k_neighbors=8), {}),
    ('pruned_pcc_k16',     dict(hyperedge_mode='pruned', metric='pcc', k_neighbors=16), {}),
    ('pruned_plv_k8',      dict(hyperedge_mode='pruned', metric='plv', k_neighbors=8), {}),
    ('hybrid_pcc_k8',      dict(hyperedge_mode='hybrid', metric='pcc', k_neighbors=8), {}),
    ('hybrid+reg',         dict(hyperedge_mode='hybrid', metric='pcc', k_neighbors=8, dropout=0.7), dict(weight_decay=1e-2, label_smoothing=0.1)),
    ('small+pruned+reg',   dict(hyperedge_mode='pruned', metric='pcc', k_neighbors=8, d_model=32, hidden=32, n_edges=8, dropout=0.6), dict(label_smoothing=0.1)),
]
rows = []
for name, mk, tk in CONFIGS:
    df, _ = T.run_subject_dependent('dhslp', subjects=[1,2,3,4,5], pp_kwargs=C.PP_MINIMAL,
                                    model_kwargs=mk, train_kwargs={**BASE_TK, **tk}, verbose=False)
    rows.append({'config': name, 'test': df.test_acc.mean(), 'train': df.train_acc.mean(),
                 'gap': df.train_acc.mean()-df.test_acc.mean()})
    print(f"{name:20s} test={rows[-1]['test']:.3f}  train={rows[-1]['train']:.3f}  gap={rows[-1]['gap']:+.3f}")
sweep = pd.DataFrame(rows).set_index('config').sort_values('test', ascending=False)
print('\nchance =', C.CHANCE_LEVEL); sweep.round(3)

## §2 — Config migliore su TUTTI i 15 soggetti (subject-dependent)
Prendi la config in cima e valutala per intero (più epoche).

In [ ]:
# imposta qui la config vincente dallo sweep (esempio: pruned_pcc_k8)
BEST_MK = dict(hyperedge_mode='pruned', metric='pcc', k_neighbors=8)   # <-- adatta al vincitore §1
BEST_TK = dict(epochs=200, patience=30, lr=1e-3, batch_size=32, label_smoothing=0.1)
df_best, res_best = T.run_subject_dependent('dhslp', pp_kwargs=C.PP_MINIMAL,
                                            model_kwargs=BEST_MK, train_kwargs=BEST_TK)
T.save_metrics(df_best, 'dhslp_maxed')
print(f"DHSLP maxed subject-dependent: {df_best.test_acc.mean():.3f} ± {df_best.test_acc.std():.3f}")
T.plot_per_subject(df_best, 'dhslp_maxed'); plt.show()

## §3 — Config migliore su mixed e independent

In [ ]:
df_mix, _ = T.run_subject_mixed('dhslp', pp_kwargs=C.PP_MINIMAL, model_kwargs=BEST_MK, train_kwargs=BEST_TK)
df_ind, _ = T.run_subject_independent('dhslp', mode='holdout', pp_kwargs=C.PP_MINIMAL, model_kwargs=BEST_MK, train_kwargs=BEST_TK)
print('mixed:', df_mix.loc['ALL','test_acc'], '| independent(holdout):', df_ind.iloc[0]['test_acc'])

## §4 — Confronto con il tetto delle ConvNet e conclusioni

In [ ]:
compare = pd.Series({
    'DHSLP originale (dep)': 0.241,
    'DHSLP maxed (dep)':     df_best.test_acc.mean(),
    'EEGNet (dep)':          0.555,
    'ShallowNet (dep)':      0.575,
}).round(3)
print('Subject-dependent, chance', C.CHANCE_LEVEL)
print(compare.to_string())
delta = df_best.test_acc.mean() - 0.241
print(f'\nMiglioramento DHSLP: {delta:+.3f}')
if df_best.test_acc.mean() > 0.45:
    print('=> DHSLP ora competitivo con le ConvNet: risultato forte.')
elif delta > 0.05:
    print('=> DHSLP migliorato ma ancora sotto le ConvNet: il pruned/reg aiuta, non ribalta.')
else:
    print('=> DHSLP resta a chance: conferma che la connettivita non porta info sulla parola.')

### Nota
Qualunque sia l'esito, è un risultato di tesi onesto:
- Se DHSLP maxed sale verso le ConvNet → il pruned + regolarizzazione funziona, DHSLP diventa competitivo.
- Se resta sotto → conferma quantitativa che **la struttura di connettività non aggiunge informazione
  sulla parola** (word NMI≈0), coerente con tutto il resto dell'analisi. In tesi: 'abbiamo massimizzato
  DHSLP con ipergrafi pruned e regolarizzazione; il tetto resta sotto le ConvNet end-to-end sul raw'.